# Adrianic v4.9 — Symmetric Decoy / Anchored Consequence Arbitration

Structurally symmetric competing identity paths. Both make stable opaque forecasts. Only downstream forecast-vs-observation agreement can distinguish them. Includes consequence reversal and symmetric anchor noise.

In [ ]:
from __future__ import annotations

import argparse
import hashlib
import json
import math
import random
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


ROOT = Path("/mnt/data")
OUT = ROOT / "adrianic_v49_symmetric_consequence_results"
OUT.mkdir(parents=True, exist_ok=True)

DEPTHS = (4, 7, 9, 15)


# =============================================================================
# Opaque identities
# =============================================================================

def opaque_token(seed: int, i: int, prefix: str) -> str:
    n = int(seed) * 1_000_000 + int(i)
    chars = []
    while True:
        chars.append(chr(ord("a") + (n % 26)))
        n = n // 26 - 1
        if n < 0:
            break
    return prefix + "".join(reversed(chars))


# =============================================================================
# Symmetric world
# =============================================================================

@dataclass
class ChainSpec:
    chain_id: str
    depth: int
    source: str
    lane_a: list[str]
    lane_b: list[str]

    forecast_a: str
    forecast_b: str
    mismatch_a: str
    mismatch_b: str

    preferred_pre: str
    preferred_post: str


class SymmetricConsequenceWorld:
    """
    Two graph routes are deliberately structurally indistinguishable.

    Both terminals also make perfectly stable opaque forecasts.

    The ONLY informative asymmetry is external consequence agreement:
      preferred terminal: forecast == observation
      other terminal:     forecast != observation

    After reversal, agreement swaps while graph and forecasts stay fixed.
    """

    def __init__(
        self,
        seed: int,
        depths=DEPTHS,
        chains_per_depth: int = 12,
    ):
        self.seed = int(seed)
        self.depths = tuple(int(d) for d in depths)
        self.chains_per_depth = int(chains_per_depth)
        self.rng = random.Random(seed + 4900)

        self.structural_events: list[dict] = []
        self.chains: list[ChainSpec] = []

        self._build()

    def _emit(self, phase, spec_id, lane, u, v, kind):
        self.structural_events.append({
            "phase": phase,
            "chain_id": spec_id,
            "lane": lane,
            "u": u,
            "v": v,
            "kind": kind,
        })

    def _build(self):
        i = 0

        for depth in self.depths:
            for ci in range(self.chains_per_depth):
                source = opaque_token(self.seed, i, "s")
                i += 1

                lane_a = [source] + [
                    opaque_token(self.seed, i + j, "a")
                    for j in range(depth)
                ]
                i += depth

                lane_b = [source] + [
                    opaque_token(self.seed, i + j, "b")
                    for j in range(depth)
                ]
                i += depth

                # Both hypotheses make stable, distinct opaque forecasts.
                forecast_a = opaque_token(self.seed, i, "p")
                forecast_b = opaque_token(self.seed, i + 1, "p")
                mismatch_a = opaque_token(self.seed, i + 2, "m")
                mismatch_b = opaque_token(self.seed, i + 3, "m")
                i += 4

                if self.rng.random() < 0.5:
                    preferred_pre = lane_a[-1]
                    preferred_post = lane_b[-1]
                else:
                    preferred_pre = lane_b[-1]
                    preferred_post = lane_a[-1]

                spec = ChainSpec(
                    chain_id=f"d{depth}_{ci}",
                    depth=depth,
                    source=source,
                    lane_a=lane_a,
                    lane_b=lane_b,
                    forecast_a=forecast_a,
                    forecast_b=forecast_b,
                    mismatch_a=mismatch_a,
                    mismatch_b=mismatch_b,
                    preferred_pre=preferred_pre,
                    preferred_post=preferred_post,
                )
                self.chains.append(spec)
                self._add_symmetric_graph(spec)

    def _add_symmetric_graph(self, spec: ChainSpec):
        """
        Exact mirrored structure on A and B.

        Shortcuts/cycles/cross-links are also mirrored.
        """
        d = spec.depth

        for phase in ("early", "mid", "late"):
            for j in range(d):
                self._emit(
                    phase, spec.chain_id, "A",
                    spec.lane_a[j], spec.lane_a[j + 1],
                    "main",
                )
                self._emit(
                    phase, spec.chain_id, "B",
                    spec.lane_b[j], spec.lane_b[j + 1],
                    "main",
                )

                if j + 2 <= d and j % 3 == 0:
                    self._emit(
                        phase, spec.chain_id, "A",
                        spec.lane_a[j], spec.lane_a[j + 2],
                        "shortcut",
                    )
                    self._emit(
                        phase, spec.chain_id, "B",
                        spec.lane_b[j], spec.lane_b[j + 2],
                        "shortcut",
                    )

                if j >= 2 and j % 4 == 2:
                    self._emit(
                        phase, spec.chain_id, "A",
                        spec.lane_a[j], spec.lane_a[j - 1],
                        "cycle",
                    )
                    self._emit(
                        phase, spec.chain_id, "B",
                        spec.lane_b[j], spec.lane_b[j - 1],
                        "cycle",
                    )

            for j in range(1, d):
                if j % 3 == 1:
                    self._emit(
                        phase, spec.chain_id, "X",
                        spec.lane_a[j], spec.lane_b[j + 1],
                        "cross",
                    )
                    self._emit(
                        phase, spec.chain_id, "X",
                        spec.lane_b[j], spec.lane_a[j + 1],
                        "cross",
                    )

    def forecast(self, spec: ChainSpec, terminal: str) -> str:
        if terminal == spec.lane_a[-1]:
            return spec.forecast_a
        if terminal == spec.lane_b[-1]:
            return spec.forecast_b
        raise KeyError(terminal)

    def observed_consequence(
        self,
        spec: ChainSpec,
        terminal: str,
        post_reversal: bool,
        noise: float = 0.0,
        rng: random.Random | None = None,
    ) -> str:
        """
        Both routes have stable consequences.

        Only one route's forecast agrees with external observation.
        """
        preferred = (
            spec.preferred_post
            if post_reversal
            else spec.preferred_pre
        )

        if rng is None:
            rng = self.rng

        correct = (terminal == preferred)

        # Symmetric anchor noise flips whether the consequence agrees with the
        # route's forecast. At noise=.30, preferred routes agree ~70% of the
        # time and alternatives ~30% of the time.
        if rng.random() < float(noise):
            correct = not correct

        if terminal == spec.lane_a[-1]:
            return (
                spec.forecast_a
                if correct
                else spec.mismatch_a
            )

        if terminal == spec.lane_b[-1]:
            return (
                spec.forecast_b
                if correct
                else spec.mismatch_b
            )

        raise KeyError(terminal)

    def symmetry_audit(self) -> pd.DataFrame:
        rows = []

        for spec in self.chains:
            ev = pd.DataFrame([
                e for e in self.structural_events
                if e["chain_id"] == spec.chain_id
            ])

            for lane in ("A", "B"):
                lane_ev = ev[ev.lane == lane]
                main = lane_ev[lane_ev.kind == "main"]

                rows.append({
                    "chain_id": spec.chain_id,
                    "depth": spec.depth,
                    "lane": lane,
                    "event_count": len(lane_ev),
                    "main_event_count": len(main),
                    "unique_edge_count": len(
                        set(zip(lane_ev.u, lane_ev.v))
                    ),
                    "main_unique_edges": len(
                        set(zip(main.u, main.v))
                    ),
                    "phase_count": lane_ev.phase.nunique(),
                    "forecast_present": 1,
                })

        return pd.DataFrame(rows)


# =============================================================================
# Equal structural graph
# =============================================================================

class EqualStructureGraph:
    def __init__(self, events: list[dict]):
        self.edge_count = Counter()
        self.out = defaultdict(dict)

        for e in events:
            self.edge_count[(e["u"], e["v"])] += 1

        max_count = max(
            self.edge_count.values(),
            default=1,
        )

        for (u, v), c in self.edge_count.items():
            self.out[u][v] = float(c / max_count)

    def shortest_distance(
        self,
        source: str,
        target: str,
        max_depth: int = 30,
    ):
        q = deque([(source, 0)])
        seen = {source}

        while q:
            node, depth = q.popleft()

            if node == target:
                return depth

            if depth >= max_depth:
                continue

            for nxt in self.out.get(node, {}):
                if nxt not in seen:
                    seen.add(nxt)
                    q.append((nxt, depth + 1))

        return None

    def paths_by_depth(
        self,
        source: str,
        max_depth: int,
        beam_width: int = 8192,
    ):
        by_depth = defaultdict(list)
        paths = [([source], 1.0)]

        for depth in range(1, max_depth + 1):
            nxt = []

            for path, score in paths:
                u = path[-1]

                for v, w in self.out.get(u, {}).items():
                    if v in path:
                        continue

                    nxt.append(
                        (
                            path + [v],
                            score * w,
                        )
                    )

            if len(nxt) > beam_width:
                nxt.sort(
                    key=lambda x: x[1],
                    reverse=True,
                )
                nxt = nxt[:beam_width]

            paths = nxt
            by_depth[depth] = list(paths)

            if not paths:
                break

        return by_depth

    def structural_scores(
        self,
        source: str,
        depth: int,
        candidates: list[str],
    ):
        paths = self.paths_by_depth(
            source,
            depth,
        ).get(depth, [])

        scores = {c: 0.0 for c in candidates}

        for path, score in paths:
            end = path[-1]

            if end in scores:
                scores[end] = max(
                    scores[end],
                    score,
                )

        mx = max(scores.values(), default=0.0)

        if mx > 0:
            scores = {
                k: float(v / mx)
                for k, v in scores.items()
            }

        return scores


# =============================================================================
# Consequence evidence
# =============================================================================

class AnchorReliability:
    """
    Generic prediction-error trust.

    The trust layer sees only:
      predicted opaque outcome
      observed opaque outcome

    It does NOT know which terminal is "true."
    """

    def __init__(
        self,
        lr: float = 0.15,
        initial: float = 0.5,
    ):
        self.lr = float(lr)
        self.initial = float(initial)
        self.rho = defaultdict(
            lambda: self.initial
        )
        self.n = Counter()

    def update(
        self,
        node: str,
        prediction: str,
        observation: str,
    ):
        q = 1.0 if prediction == observation else 0.0

        self.rho[node] = (
            (1.0 - self.lr) * self.rho[node]
            + self.lr * q
        )
        self.n[node] += 1

    def trust(self, node: str) -> float:
        return float(self.rho[node])

    def snapshot(self):
        return dict(self.rho)


class CoherenceOnlyTrust:
    """
    Control: asks only whether consequence observations are self-consistent.

    Both terminals are perfectly stable in v4.9, so this lane should NOT know
    which route agrees with reality.
    """

    def __init__(self):
        self.symbol = {}
        self.consistent = defaultdict(lambda: True)
        self.support = Counter()

    def update(self, node: str, observation: str):
        if node not in self.symbol:
            self.symbol[node] = observation
        elif self.symbol[node] != observation:
            self.consistent[node] = False

        self.support[node] += 1

    def trust(self, node: str) -> float:
        if self.support[node] == 0:
            return 0.0

        support = 1.0 - math.exp(
            -self.support[node] / 5.0
        )
        return (
            support
            if self.consistent[node]
            else 0.0
        )


class FrozenTrust:
    def __init__(self, snapshot):
        self.map = dict(snapshot)

    def trust(self, node):
        return float(self.map.get(node, 0.5))


class ShuffledTrust:
    def __init__(
        self,
        snapshot,
        nodes,
        seed,
    ):
        vals = [
            snapshot.get(n, 0.5)
            for n in nodes
        ]
        rng = random.Random(seed)
        rng.shuffle(vals)
        self.map = dict(zip(nodes, vals))

    def trust(self, node):
        return float(self.map.get(node, 0.5))


# =============================================================================
# Arbitration
# =============================================================================

def deterministic_winner(
    scores: dict[str, float],
    candidates: list[str],
):
    ranked = sorted(
        candidates,
        key=lambda c: scores.get(c, 0.0),
        reverse=True,
    )

    if len(ranked) < 2:
        return ranked[0] if ranked else None

    a, b = ranked[0], ranked[1]
    sa = float(scores.get(a, 0.0))
    sb = float(scores.get(b, 0.0))

    if abs(sa - sb) > 1e-12:
        return a

    # Hashed tie break prevents permanent lane bias.
    h = int(
        hashlib.sha256(
            "|".join(sorted(candidates)).encode()
        ).hexdigest()[:8],
        16,
    )
    return sorted(candidates)[h % len(candidates)]


def fused_scores(
    graph: EqualStructureGraph,
    trust_model,
    source: str,
    depth: int,
    candidates: list[str],
):
    structural = graph.structural_scores(
        source,
        depth,
        candidates,
    )

    return {
        c: float(
            math.sqrt(
                max(0.0, structural.get(c, 0.0))
                * max(0.0, trust_model.trust(c))
            )
        )
        for c in candidates
    }


def endogenous_select(
    graph: EqualStructureGraph,
    trust_model,
    source: str,
    candidates: list[str],
    max_depth: int,
):
    """
    No nominal depth supplied.

    Search all depths and choose the strongest consequence-supported endpoint.
    If a real shortcut reaches the same terminal earlier, that earlier distance
    is allowed: path length is defined by the actual graph, not by the generator's
    nominal main-lane length.
    """
    by_depth = graph.paths_by_depth(
        source,
        max_depth,
    )

    best = None
    trace = []

    for depth in range(1, max_depth + 1):
        paths = by_depth.get(depth, [])
        structural = {c: 0.0 for c in candidates}

        for path, score in paths:
            end = path[-1]

            if end in structural:
                structural[end] = max(
                    structural[end],
                    score,
                )

        scores = {
            c: math.sqrt(
                max(0.0, structural[c])
                * max(0.0, trust_model.trust(c))
            )
            for c in candidates
        }

        ranked = sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True,
        )

        top = ranked[0] if ranked else (None, 0.0)
        second = ranked[1] if len(ranked) > 1 else (None, 0.0)

        margin = max(
            0.0,
            top[1] - second[1],
        )

        utility = float(
            top[1] * (0.5 + 0.5 * margin)
        )

        row = {
            "depth": depth,
            "top": top[0],
            "top_score": float(top[1]),
            "second_score": float(second[1]),
            "stop_utility": utility,
            "path_count": len(paths),
        }
        trace.append(row)

        if (
            best is None
            or utility > best["stop_utility"]
            + 1e-12
            or (
                abs(
                    utility
                    - best["stop_utility"]
                ) <= 1e-12
                and depth < best["depth"]
            )
        ):
            best = row

    return best, trace


# =============================================================================
# Experiment
# =============================================================================

def run_seed(
    seed: int,
    depths=DEPTHS,
    chains_per_depth: int = 12,
    pre_episodes: int = 24,
    reversal_episodes: int = 24,
    trust_lr: float = 0.15,
    anchor_noise: float = 0.0,
):
    world = SymmetricConsequenceWorld(
        seed=seed,
        depths=depths,
        chains_per_depth=chains_per_depth,
    )

    graph = EqualStructureGraph(
        world.structural_events
    )

    anchor = AnchorReliability(
        lr=trust_lr,
    )
    coherence = CoherenceOnlyTrust()
    consequence_rng = random.Random(seed + 49991)

    terminals = []

    for spec in world.chains:
        terminals.extend([
            spec.lane_a[-1],
            spec.lane_b[-1],
        ])

    # ------------------------------------------------------------
    # Pre-reversal evidence
    # ------------------------------------------------------------
    for _ in range(pre_episodes):
        for spec in world.chains:
            for terminal in (
                spec.lane_a[-1],
                spec.lane_b[-1],
            ):
                pred = world.forecast(
                    spec,
                    terminal,
                )
                obs = world.observed_consequence(
                    spec,
                    terminal,
                    post_reversal=False,
                    noise=anchor_noise,
                    rng=consequence_rng,
                )

                anchor.update(
                    terminal,
                    pred,
                    obs,
                )
                coherence.update(
                    terminal,
                    obs,
                )

    frozen = FrozenTrust(
        anchor.snapshot()
    )
    shuffled = ShuffledTrust(
        anchor.snapshot(),
        terminals,
        seed + 49091,
    )

    pre_rows = []

    for spec in world.chains:
        candidates = [
            spec.lane_a[-1],
            spec.lane_b[-1],
        ]

        nominal = spec.depth
        shortest_a = graph.shortest_distance(
            spec.source,
            candidates[0],
            max_depth=max(depths),
        )
        shortest_b = graph.shortest_distance(
            spec.source,
            candidates[1],
            max_depth=max(depths),
        )

        structural = graph.structural_scores(
            spec.source,
            nominal,
            candidates,
        )
        structure_choice = deterministic_winner(
            structural,
            candidates,
        )

        coherence_scores = fused_scores(
            graph,
            coherence,
            spec.source,
            nominal,
            candidates,
        )
        coherence_choice = deterministic_winner(
            coherence_scores,
            candidates,
        )

        anchor_scores = fused_scores(
            graph,
            anchor,
            spec.source,
            nominal,
            candidates,
        )
        anchor_choice = deterministic_winner(
            anchor_scores,
            candidates,
        )

        shuffled_scores = fused_scores(
            graph,
            shuffled,
            spec.source,
            nominal,
            candidates,
        )
        shuffled_choice = deterministic_winner(
            shuffled_scores,
            candidates,
        )

        stop, trace = endogenous_select(
            graph,
            anchor,
            spec.source,
            candidates,
            max_depth=max(depths),
        )

        chosen_shortest = (
            shortest_a
            if stop["top"] == candidates[0]
            else shortest_b
        )

        other = (
            spec.preferred_post
        )

        pre_rows.append({
            "seed": seed,
            "phase": "pre",
            "chain_id": spec.chain_id,
            "depth": nominal,
            "preferred": spec.preferred_pre,

            "structure_choice": structure_choice,
            "coherence_choice": coherence_choice,
            "anchor_choice": anchor_choice,
            "shuffled_choice": shuffled_choice,
            "endogenous_choice": stop["top"],

            "structure_correct": int(
                structure_choice
                == spec.preferred_pre
            ),
            "coherence_correct": int(
                coherence_choice
                == spec.preferred_pre
            ),
            "anchor_correct": int(
                anchor_choice
                == spec.preferred_pre
            ),
            "shuffled_correct": int(
                shuffled_choice
                == spec.preferred_pre
            ),
            "endogenous_correct": int(
                stop["top"]
                == spec.preferred_pre
            ),

            "selected_depth": int(
                stop["depth"]
            ),
            "preferred_shortest_depth": int(
                chosen_shortest
            ),
            "shortest_depth_correct": int(
                stop["depth"]
                == chosen_shortest
            ),

            "binding_margin": float(
                anchor_scores[
                    spec.preferred_pre
                ]
                - anchor_scores[other]
            ),

            "preferred_trust": float(
                anchor.trust(
                    spec.preferred_pre
                )
            ),
            "other_trust": float(
                anchor.trust(other)
            ),

            "coherence_gap": float(
                coherence.trust(
                    spec.preferred_pre
                )
                - coherence.trust(other)
            ),

            "shortest_a": shortest_a,
            "shortest_b": shortest_b,
            "stop_trace": json.dumps(trace),
        })

    # ------------------------------------------------------------
    # Consequence reversal with graph and forecasts frozen
    # ------------------------------------------------------------
    first_switch = {
        spec.chain_id: None
        for spec in world.chains
    }

    post_rows = []

    for ep in range(1, reversal_episodes + 1):
        for spec in world.chains:
            for terminal in (
                spec.lane_a[-1],
                spec.lane_b[-1],
            ):
                pred = world.forecast(
                    spec,
                    terminal,
                )
                obs = world.observed_consequence(
                    spec,
                    terminal,
                    post_reversal=True,
                    noise=anchor_noise,
                    rng=consequence_rng,
                )

                anchor.update(
                    terminal,
                    pred,
                    obs,
                )

        for spec in world.chains:
            candidates = [
                spec.lane_a[-1],
                spec.lane_b[-1],
            ]

            adaptive_scores = fused_scores(
                graph,
                anchor,
                spec.source,
                spec.depth,
                candidates,
            )
            adaptive_choice = deterministic_winner(
                adaptive_scores,
                candidates,
            )

            if (
                first_switch[spec.chain_id] is None
                and adaptive_choice
                == spec.preferred_post
            ):
                first_switch[spec.chain_id] = ep

    # final post-reversal evaluation
    for spec in world.chains:
        candidates = [
            spec.lane_a[-1],
            spec.lane_b[-1],
        ]

        adaptive_scores = fused_scores(
            graph,
            anchor,
            spec.source,
            spec.depth,
            candidates,
        )
        adaptive_choice = deterministic_winner(
            adaptive_scores,
            candidates,
        )

        frozen_scores = fused_scores(
            graph,
            frozen,
            spec.source,
            spec.depth,
            candidates,
        )
        frozen_choice = deterministic_winner(
            frozen_scores,
            candidates,
        )

        stop, trace = endogenous_select(
            graph,
            anchor,
            spec.source,
            candidates,
            max_depth=max(depths),
        )

        chosen_shortest = graph.shortest_distance(
            spec.source,
            stop["top"],
            max_depth=max(depths),
        )

        other = spec.preferred_pre

        post_rows.append({
            "seed": seed,
            "phase": "post",
            "chain_id": spec.chain_id,
            "depth": spec.depth,
            "preferred": spec.preferred_post,

            "adaptive_choice": adaptive_choice,
            "frozen_choice": frozen_choice,
            "endogenous_choice": stop["top"],

            "adaptive_correct": int(
                adaptive_choice
                == spec.preferred_post
            ),
            "frozen_correct": int(
                frozen_choice
                == spec.preferred_post
            ),
            "endogenous_correct": int(
                stop["top"]
                == spec.preferred_post
            ),

            "selected_depth": int(
                stop["depth"]
            ),
            "chosen_shortest_depth": int(
                chosen_shortest
            ),
            "shortest_depth_correct": int(
                stop["depth"]
                == chosen_shortest
            ),

            "binding_margin": float(
                adaptive_scores[
                    spec.preferred_post
                ]
                - adaptive_scores[other]
            ),

            "preferred_trust": float(
                anchor.trust(
                    spec.preferred_post
                )
            ),
            "other_trust": float(
                anchor.trust(other)
            ),

            "switch_episode": int(
                first_switch[spec.chain_id]
                if first_switch[spec.chain_id]
                is not None
                else reversal_episodes + 1
            ),

            "stop_trace": json.dumps(trace),
        })

    pre = pd.DataFrame(pre_rows)
    post = pd.DataFrame(post_rows)
    audit = world.symmetry_audit()

    # ------------------------------------------------------------
    # Exact structural symmetry audit
    # ------------------------------------------------------------
    p = audit.pivot(
        index=["chain_id", "depth"],
        columns="lane",
        values=[
            "event_count",
            "main_event_count",
            "unique_edge_count",
            "main_unique_edges",
            "phase_count",
            "forecast_present",
        ],
    )

    symmetry_exact = True

    for metric in [
        "event_count",
        "main_event_count",
        "unique_edge_count",
        "main_unique_edges",
        "phase_count",
        "forecast_present",
    ]:
        symmetry_exact &= bool(
            np.array_equal(
                p[(metric, "A")].to_numpy(),
                p[(metric, "B")].to_numpy(),
            )
        )

    summaries = []

    for depth in depths:
        pg = pre[pre.depth == depth]
        qg = post[post.depth == depth]

        summaries.append({
            "seed": seed,
            "depth": int(depth),
            "anchor_noise": float(anchor_noise),
            "symmetry_exact": symmetry_exact,

            "pre_structure_accuracy": float(
                pg.structure_correct.mean()
            ),
            "pre_coherence_accuracy": float(
                pg.coherence_correct.mean()
            ),
            "pre_shuffled_anchor_accuracy": float(
                pg.shuffled_correct.mean()
            ),
            "pre_anchor_accuracy": float(
                pg.anchor_correct.mean()
            ),
            "pre_endogenous_accuracy": float(
                pg.endogenous_correct.mean()
            ),
            "pre_shortest_stop_accuracy": float(
                pg.shortest_depth_correct.mean()
            ),

            "pre_binding_margin": float(
                pg.binding_margin.mean()
            ),
            "pre_anchor_trust_gap": float(
                (
                    pg.preferred_trust
                    - pg.other_trust
                ).mean()
            ),
            "pre_coherence_gap": float(
                pg.coherence_gap.mean()
            ),

            "post_adaptive_accuracy": float(
                qg.adaptive_correct.mean()
            ),
            "post_frozen_accuracy": float(
                qg.frozen_correct.mean()
            ),
            "post_endogenous_accuracy": float(
                qg.endogenous_correct.mean()
            ),
            "post_shortest_stop_accuracy": float(
                qg.shortest_depth_correct.mean()
            ),

            "post_binding_margin": float(
                qg.binding_margin.mean()
            ),
            "post_anchor_trust_gap": float(
                (
                    qg.preferred_trust
                    - qg.other_trust
                ).mean()
            ),

            "switch_episode_median": float(
                qg.switch_episode.median()
            ),

            "selected_depth_pre_median": float(
                pg.selected_depth.median()
            ),
            "selected_depth_post_median": float(
                qg.selected_depth.median()
            ),
        })

    return (
        pd.DataFrame(summaries),
        pre,
        post,
        audit,
    )


def run_many(
    seeds: int = 8,
    chains_per_depth: int = 12,
    pre_episodes: int = 24,
    reversal_episodes: int = 24,
    trust_lr: float = 0.15,
    anchor_noise: float = 0.0,
):
    summaries = []
    pres = []
    posts = []
    audits = []

    for seed in range(seeds):
        print("seed", seed)

        s, pre, post, audit = run_seed(
            seed=seed,
            chains_per_depth=chains_per_depth,
            pre_episodes=pre_episodes,
            reversal_episodes=reversal_episodes,
            trust_lr=trust_lr,
            anchor_noise=anchor_noise,
        )

        summaries.append(s)
        pres.append(pre)
        posts.append(post)
        audit["seed"] = seed
        audits.append(audit)

        print(
            s[[
                "depth",
                "pre_structure_accuracy",
                "pre_coherence_accuracy",
                "pre_shuffled_anchor_accuracy",
                "pre_anchor_accuracy",
                "pre_endogenous_accuracy",
                "post_adaptive_accuracy",
                "post_frozen_accuracy",
                "switch_episode_median",
            ]].to_string(index=False)
        )

    return (
        pd.concat(summaries, ignore_index=True),
        pd.concat(pres, ignore_index=True),
        pd.concat(posts, ignore_index=True),
        pd.concat(audits, ignore_index=True),
    )


def make_checks(summary):
    checks = {}

    for depth in sorted(summary.depth.unique()):
        g = summary[summary.depth == depth]

        checks[
            f"d{depth}_symmetry_exact_seed_fraction"
        ] = float(np.mean(
            g.symmetry_exact
        ))

        checks[
            f"d{depth}_anchor_beats_structure_seed_fraction"
        ] = float(np.mean(
            g.pre_anchor_accuracy
            > g.pre_structure_accuracy
        ))

        checks[
            f"d{depth}_anchor_beats_coherence_seed_fraction"
        ] = float(np.mean(
            g.pre_anchor_accuracy
            > g.pre_coherence_accuracy
        ))

        checks[
            f"d{depth}_anchor_beats_shuffle_seed_fraction"
        ] = float(np.mean(
            g.pre_anchor_accuracy
            > g.pre_shuffled_anchor_accuracy
        ))

        checks[
            f"d{depth}_pre_anchor_accuracy_median"
        ] = float(np.median(
            g.pre_anchor_accuracy
        ))

        checks[
            f"d{depth}_post_adaptive_accuracy_median"
        ] = float(np.median(
            g.post_adaptive_accuracy
        ))

        checks[
            f"d{depth}_post_frozen_accuracy_median"
        ] = float(np.median(
            g.post_frozen_accuracy
        ))

        checks[
            f"d{depth}_switch_episode_median"
        ] = float(np.median(
            g.switch_episode_median
        ))

        checks[
            f"d{depth}_pre_shortest_stop_accuracy_median"
        ] = float(np.median(
            g.pre_shortest_stop_accuracy
        ))

        checks[
            f"d{depth}_post_shortest_stop_accuracy_median"
        ] = float(np.median(
            g.post_shortest_stop_accuracy
        ))

        checks[
            f"d{depth}_coherence_gap_abs_median"
        ] = float(np.median(
            np.abs(g.pre_coherence_gap)
        ))

    return checks


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--seeds", type=int, default=8)
    parser.add_argument(
        "--chains-per-depth",
        type=int,
        default=12,
    )
    parser.add_argument(
        "--pre-episodes",
        type=int,
        default=24,
    )
    parser.add_argument(
        "--reversal-episodes",
        type=int,
        default=24,
    )
    parser.add_argument(
        "--trust-lr",
        type=float,
        default=0.15,
    )
    parser.add_argument(
        "--anchor-noise",
        type=float,
        default=0.0,
    )
    args = parser.parse_args(argv)

    summary, pre, post, audit = run_many(
        seeds=args.seeds,
        chains_per_depth=args.chains_per_depth,
        pre_episodes=args.pre_episodes,
        reversal_episodes=args.reversal_episodes,
        trust_lr=args.trust_lr,
        anchor_noise=args.anchor_noise,
    )

    tag = f"{args.seeds}seed"

    summary.to_csv(
        OUT / f"{tag}_summary.csv",
        index=False,
    )
    pre.to_csv(
        OUT / f"{tag}_pre_details.csv",
        index=False,
    )
    post.to_csv(
        OUT / f"{tag}_post_details.csv",
        index=False,
    )
    audit.to_csv(
        OUT / f"{tag}_symmetry_audit.csv",
        index=False,
    )

    checks = make_checks(summary)

    (
        OUT / f"{tag}_checks.json"
    ).write_text(
        json.dumps(checks, indent=2)
    )

    med = summary.groupby(
        "depth"
    ).median(numeric_only=True)

    report = f"""Adrianic v4.9 — Symmetric Decoy / Anchored Consequence Arbitration

Seeds: {args.seeds}
Depths: {list(DEPTHS)}
Chains per depth: {args.chains_per_depth}
Anchor noise: {args.anchor_noise}

Question
========
When competing identity paths are structurally symmetric AND both make stable
downstream forecasts, can external consequence agreement decide which path
deserves belief?

Structural symmetry
===================
Both candidate routes have matched:
- path construction
- edge recurrence
- temporal phases
- mirrored shortcuts
- mirrored cycles
- mirrored cross-links
- structural observation count

Which route is externally confirmed is randomized independently per chain.

Consequence symmetry
====================
Both terminals make stable opaque forecasts.
Both terminals also receive stable opaque observations.

Therefore:
- structure alone cannot distinguish them;
- recurrence/persistence cannot distinguish them;
- consequence coherence alone cannot distinguish them.

The only useful evidence is whether a route's forecast matches its later
external observation.

Reversal
========
After the first evidence phase, the structural graph and forecasts are frozen.

Only which route agrees with external consequence is reversed.

An adaptive trust system should switch.
A frozen trust system should fail.

Median metrics
==============
{med.to_string()}

Checks
======
{json.dumps(checks, indent=2)}

Stopping diagnostic
===================
The graph contains real shortcuts. Therefore the generator's nominal chain
length is NOT treated as the correct stopping depth.

The honest stopping target is the shortest actually supported path to the
chosen terminal. Endogenous stopping is evaluated against that graph-derived
distance.

Interpretation boundary
=======================
A positive result supports consequence-sensitive structural arbitration:
belief changes because observed consequences agree or disagree with a path's
forecast.

It does not establish semantic truth, unrestricted planning, or internally
learned formal logic.
"""

    (
        OUT / f"{tag}_REPORT.txt"
    ).write_text(report)

    print(report)




## Clean 8-seed run

```python
main(["--seeds", "8", "--anchor-noise", "0.0"])
```

In [ ]:
main(["--seeds", "8", "--anchor-noise", "0.0"])

## 15% anchor-noise stress

```python
main(["--seeds", "8", "--anchor-noise", "0.15"])
```

In [ ]:
main(["--seeds", "8", "--anchor-noise", "0.15"])

## 30% anchor-noise stress

```python
main(["--seeds", "8", "--anchor-noise", "0.30"])
```

In [ ]:
main(["--seeds", "8", "--anchor-noise", "0.30"])

## Verified local interpretation

# Adrianic v4.9 — Symmetric Decoy / Anchored Consequence Arbitration

## Verified local result

This version was deliberately made stricter than the first coherence-only draft.

The decisive test is **not** "stable consequence vs noisy consequence."

Both candidate routes now have:

- exact mirrored topology,
- matched edge recurrence and temporal exposure,
- matched shortcuts, cycles and cross-links,
- stable opaque forecasts,
- stable opaque downstream observations.

So structure, recurrence, persistence, and mere consequence coherence cannot tell
the routes apart.

The only informative variable is:

> **does the route's forecast agree with the later external observation?**

Which lane agrees is randomized independently per chain.

After a reversal point, the graph and forecasts are frozen and only external
consequence agreement swaps.

## Verification scale

- 8 seeds
- depths 4, 7, 9, 15
- 12 independent chains per depth
- 384 probe chains per noise condition
- anchor noise tested at 0%, 15%, and 30%
- exact A/B structural symmetry audit: **True**

## Pooled results

```text
 noise_pct  n_pre_cases  structure_accuracy  coherence_accuracy  shuffled_anchor_accuracy  anchor_accuracy  endogenous_accuracy_pre  adaptive_accuracy_post  frozen_accuracy_post  endogenous_accuracy_post  shortest_stop_accuracy_pre  shortest_stop_accuracy_post  median_switch_episode  mean_pre_binding_margin  mean_post_binding_margin
         0          384            0.515625            0.515625                  0.541667          1.00000                  1.00000                1.000000               0.00000                  1.000000                         1.0                          1.0                    5.0                 0.894349                  0.848415
        15          384            0.515625            0.518229                  0.523438          1.00000                  1.00000                1.000000               0.00000                  1.000000                         1.0                          1.0                    5.0                 0.552477                  0.527383
        30          384            0.515625            0.515625                  0.502604          0.96875                  0.96875                0.984375               0.03125                  0.984375                         1.0                          1.0                    4.0                 0.289361                  0.285549
```

## Clean consequence anchors

At 0% anchor noise:

- structure-only pooled accuracy: **0.516**
- coherence-only pooled accuracy: **0.516**
- shuffled-anchor pooled accuracy: **0.542**
- anchored consequence arbitration: **1.000**
- post-reversal adaptive accuracy: **1.000**
- post-reversal frozen accuracy: **0.000**

The adaptive controller switches to the newly supported route in a median of
**5.0 consequence observations**.

## Noisy anchors

At 15% symmetric anchor corruption:

- anchored pre-reversal accuracy: **1.000**
- adaptive post-reversal accuracy: **1.000**
- frozen post-reversal accuracy: **0.000**

At 30% symmetric anchor corruption:

- anchored pre-reversal accuracy: **0.969**
- adaptive post-reversal accuracy: **0.984**
- frozen post-reversal accuracy: **0.031**
- median switch latency: **4.0**

## Stopping correction

The graph contains genuine shortcuts. Therefore the generator's nominal main-path
length is not an honest stopping target.

For example, nominal depths 4, 7, 9 and 15 have shortest supported routes of
approximately 3, 5, 6 and 10 hops.

The endogenous selector is therefore evaluated against the graph-derived
shortest route to the chosen terminal, not against the arbitrary generator
length.

That shortest-route stopping accuracy was **1.000 pooled before and after
reversal at every tested noise level**.

## What this supports

Within this controlled structural benchmark:

1. **Structure alone cannot choose the route.**
2. **Coherence alone cannot choose the route.**
3. **Shuffling anchor trust destroys the advantage.**
4. **Observed consequence agreement selects the correct route.**
5. **When consequences reverse but structure does not, adaptive trust reverses
   preference while frozen trust fails.**
6. The effect survives **30% symmetric anchor corruption**.

That is the behavior the test was designed to isolate:

> structurally indistinguishable hypotheses are arbitrated by whether their
> predictions continue to agree with later reality.

## What it does NOT establish

This is still a synthetic, opaque-symbol structural benchmark.

It does not establish:

- semantic truth discovery from natural language,
- unrestricted logical reasoning,
- transformer-level semantic representation,
- or that the recurrent model itself learned the arbitration rule.

The important result is narrower and cleaner:

> **the deterministic Adrianic structural/trust layer can use downstream
> prediction error as the deciding evidence when identity, topology,
> persistence, and coherence are deliberately made non-informative.**

## Next gate

The next serious integration test is to replace the opaque forecast/observation
symbols with a semantic front end while keeping this arbitration mechanism
unchanged.

A transformer could propose entities, relations and forecasts; this layer would
still decide which proposed structure deserves trust from consequences rather
than from semantic confidence alone.
